# Flexor: Synthetic Dataset Generator

This notebook implements a Gradio app for generating structured, realistic synthetic datasets from natural language descriptions using open-source LLMs via HuggingFace Inference API.

---

## 1. Import Required Libraries and Dependencies
Import all necessary Python packages for the project.

In [4]:
# Imports
import os
import json
import pandas as pd
import gradio as gr
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)
HF_TOKEN = os.getenv('HUGGINGFACE_TOKEN')

assert HF_TOKEN, "HuggingFace token not found in environment variables!"


---

## 2. HuggingFace Inference API Utility
Define a function to call the chat_completion endpoint for dataset generation using instruct models (LLaMA 3, Zephyr).

In [19]:
# HuggingFace Inference API Utility

def chat_completion(prompt, model="meta-llama/Meta-Llama-3-8B-Instruct"):
    url = f"https://router.huggingface.co//v1/chat/completions"
    headers = {"Authorization": f"Bearer {HF_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 2048,
        "temperature": 0.7
    }
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

---

## 3. Dataset Generation Utility
Create a function to generate structured data from a natural language description, with optional column schema and batch generation (up to 200 rows, 20 per batch).

In [15]:
# Dataset Generation Utility

def generate_dataset(description, columns=None, num_rows=100, model="meta-llama/Meta-Llama-3-8B-Instruct"):
    rows_per_batch = 20
    batches = min((num_rows + rows_per_batch - 1) // rows_per_batch, 10)
    all_rows = []
    for batch in range(batches):
        prompt = f"Generate {rows_per_batch} rows of structured data as JSON based on this description: '{description}'."
        if columns:
            prompt += f" Columns: {columns}."
        prompt += " Respond with a JSON array of objects."
        result = chat_completion(prompt, model=model)
        try:
            batch_rows = json.loads(result)
            if isinstance(batch_rows, list):
                all_rows.extend(batch_rows)
        except Exception:
            continue
    return all_rows[:num_rows]

---

## 4. Export Utilities
Functions to export generated data as JSON or CSV.

In [16]:
# Export Utilities

def export_json(data):
    return json.dumps(data, indent=2)

def export_csv(data):
    df = pd.DataFrame(data)
    return df.to_csv(index=False)


---

## 5. Gradio UI Construction
Build a clean Gradio interface for natural language dataset definition, optional column schema, export options, and batch generation.

In [22]:
# Gradio UI Construction

def gradio_app():
    with gr.Blocks() as demo:
        gr.Markdown("## Flexor: Synthetic Dataset Generator")
        description = gr.Textbox(label="Describe your desired dataset (plain English)", lines=3)
        columns = gr.Textbox(label="Optional: Specify column schema (comma-separated)")
        num_rows = gr.Slider(20, 200, value=100, step=20, label="Number of rows")
        model = gr.Dropdown(["meta-llama/Meta-Llama-3-8B-Instruct", "HuggingFaceH4/zephyr-7b-beta"], value="meta-llama/Meta-Llama-3-8B-Instruct", label="Model")
        generate_btn = gr.Button("Generate Dataset")
        json_output = gr.Code(label="JSON Output", language="json")
        csv_output = gr.File(label="CSV Download")

        def on_generate(desc, cols, rows, model):
            data = generate_dataset(desc, cols, rows, model)
            json_str = export_json(data)
            csv_str = export_csv(data)
            csv_path = "synthetic_dataset.csv"
            with open(csv_path, "w") as f:
                f.write(csv_str)
            return json_str, csv_path

        generate_btn.click(on_generate, [description, columns, num_rows, model], [json_output, csv_output])
    return demo


Traceback (most recent call last):
  File "/workspaces/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/llm_engineering/.venv/lib/python3.12/site-packages/anyio/t

---

## 6. Main Application Runner
Combine all utilities and UI components to run the application.

In [20]:
# Main Application Runner

def main():
    demo = gradio_app()
    demo.launch()

if __name__ == '__main__':
    main()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
